In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
import pickle


In [2]:
df=pd.read_csv('Churn_Modelling (1).csv')
df.head()

In [3]:
df=df.drop(['RowNumber','CustomerId','Surname'],axis=1)
df.head()

In [4]:
df['Gender'].value_counts()

In [5]:
# using onehot encoding on gender column
lable_encoder_gender=LabelEncoder()
df['Gender'] = lable_encoder_gender.fit_transform(df['Gender'])

In [6]:
df.head()


In [7]:
df['Geography'].value_counts()

In [8]:
one_hot_encoder_geo=OneHotEncoder(sparse_output=False)
geography_encoder=one_hot_encoder_geo.fit_transform(df[['Geography']])
df.head()




In [9]:
one_hot_encoder_geo.get_feature_names_out(['Geography'])

In [10]:
geography_encoder_df=pd.DataFrame(geography_encoder,columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

In [11]:
df=pd.concat([df.drop('Geography',axis=1),geography_encoder_df],axis=1)
df.head()

In [12]:
X=df.drop(['Exited'],axis=1)
y=df['Exited']

In [13]:
y

In [14]:
# saving the pickel files for further use
with open('lable_encoder_gender.pkl','wb') as file:
    pickle.dump(lable_encoder_gender,file)

with open('onehot_encoder_geography.pkl','wb')as file:
    pickle.dump(one_hot_encoder_geo,file)

In [15]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()


In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)

In [17]:
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


In [18]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [19]:
df.head()

APPLYING ANN

In [20]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


In [21]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [22]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [23]:

opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

In [24]:
#compiling

model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [25]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard


In [26]:
tensorflow_callback = tensorflow.keras.callbacks.TensorBoard(log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"), histogram_freq=1)

In [27]:
early_stopping_callback=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [28]:
history=model.fit(
X_train,y_train,validation_data=(X_test,y_test),epochs=25,
callbacks=(tensorflow_callback,early_stopping_callback)
)

Epoch 1/25

250/250 [==============================] - 2s 6ms/step - loss: 0.3924 - accuracy: 0.8401 - val_loss: 0.3632 - val_accuracy: 0.8480
Epoch 2/25

250/250 [==============================] - 1s 4ms/step - loss: 0.3541 - accuracy: 0.8558 - val_loss: 0.3444 - val_accuracy: 0.8560
Epoch 3/25

250/250 [==============================] - 1s 3ms/step - loss: 0.3456 - accuracy: 0.8577 - val_loss: 0.3421 - val_accuracy: 0.8650
Epoch 4/25

250/250 [==============================] - 1s 3ms/step - loss: 0.3417 - accuracy: 0.8615 - val_loss: 0.3463 - val_accuracy: 0.8500
Epoch 5/25

250/250 [==============================] - 1s 3ms/step - loss: 0.3407 - accuracy: 0.8622 - val_loss: 0.3427 - val_accuracy: 0.8525
Epoch 6/25

250/250 [==============================] - 1s 3ms/step - loss: 0.3367 - accuracy: 0.8599 - val_loss: 0.3341 - val_accuracy: 0.8560
Epoch 7/25

250/250 [==============================] - 1s 3ms/step - loss: 0.3357 - accuracy: 0.8648 - val_loss: 0.3361 - val_accuracy: 0.8600

In [29]:
model.save('model.h5')

D:\ANN Classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [30]:
# loading tensorflow
%load_ext tensorboard


In [31]:
%tensorboard --logdir logs/fit